In [ ]:
import os
import pickle
import pandas as pd
from pathlib import Path
from tqdm import tqdm

# ================= 配置区 =================
# 指向你的 result 根目录，例如 './result/archive-98'
ROOT_DIR = "../result/archive-100" 
OUTPUT_FILE_LOC = "./result_100/experiment_results_PowerBI.csv"
# ==========================================

def process_experiment_data(root_path):
    root_path = Path(root_path)
    data_records = []
    
    # 递归查找所有的 .pckl 文件
    pckl_files = list(root_path.rglob("*.pckl"))
    
    print(f"🔍 寻找到 {len(pckl_files)} 个 pickle 文件，开始解析...")
    
    for pckl_file in tqdm(pckl_files, desc="Parsing Files"):
        try:
            # 1. 从目录结构解析环境参数
            seed_str = pckl_file.parent.name
            num_str = pckl_file.parent.parent.name
            task_name = pckl_file.parent.parent.parent.name
            
            # --- 过滤：无视 caltech n=767 ---
            if task_name == 'caltech' and num_str == '767':
                continue
            
            # 2. 从文件名解析策略参数
            filename = pckl_file.stem
            parts = filename.split('-')
            
            if not parts:
                continue
                
            alg = parts[0]
            
            # 建立基础字典
            record = {
                'Task': task_name,
                'Ground_Size': int(num_str),
                'Seed': int(seed_str),
                'Algorithm': alg,
                'Strategy': None,
                'Heuristic': None,
                'Sorting': None,
                'UB': None,
                'D': None,
                'Budget': None,
                'Alpha': None,
                'Model': None,
            }

            # 3. 针对不同算法格式进行解析
            if alg == 'EfficientBFS' and len(parts) >= 7:
                strategy_raw = parts[1]
                
                # 遇到带有 _LS 后缀的文件直接跳过
                if strategy_raw.endswith('_LS'):
                    continue
                
                record['Strategy'] = strategy_raw
                record['Heuristic'] = parts[2]
                record['Sorting'] = parts[3]
                record['Budget'] = float(parts[4])
                record['Alpha'] = float(parts[5])
                record['Model'] = "-".join(parts[6:])
                
            elif alg in ['Efficient', 'BFSTC'] and len(parts) >= 6:
                record['UB'] = parts[1]
                record['D'] = parts[2]
                record['Budget'] = float(parts[3])
                record['Alpha'] = float(parts[4])
                record['Model'] = "-".join(parts[5:])
                
            else:
                continue

            # 4. 读取 Pickle 内容
            with open(pckl_file, 'rb') as f:
                res = pickle.load(f)
                
            # --- 核心修改：处理 Time_s 和 TLE ---
            raw_time = res.get('time', None)
            raw_tle = res.get('TLE', False)
            
            if raw_time is not None and raw_time >= 5000:
                raw_time = 5000
                raw_tle = True
            # ------------------------------------
                
            # 5. 组装核心指标
            record.update({
                'Objective_f(S)': res.get('f(S)', None),
                'Cost_c(S)': res.get('c(S)', None),
                'Time_s': raw_time,
                'Node_Count': res.get('node_count', None),
                'Open_List_Count': res.get('open_list_count', None),
                'TLE': raw_tle,
                'Solution_Set_Size': len(res.get('S', [])) if 'S' in res else 0 
            })
            
            data_records.append(record)
            
        except Exception as e:
            print(f"❌ 解析出错 {pckl_file.name}: {e}")

    # 6. 转换为 DataFrame 并导出
    df = pd.DataFrame(data_records)
    
    if not df.empty:
        # 按照任务、算法、预算、种子排序
        df.sort_values(by=['Task', 'Algorithm', 'Budget', 'Seed'], inplace=True)
        
        # 自动创建输出目录
        Path(OUTPUT_FILE_LOC).parent.mkdir(parents=True, exist_ok=True)
        
        df.to_csv(OUTPUT_FILE_LOC, index=False, encoding='utf-8-sig')
        print(f"\n✅ 数据处理完毕！共提取 {len(df)} 条有效记录。")
        print(f"💾 已保存至: {OUTPUT_FILE_LOC}")
    else:
        print("\n⚠️ 未提取到任何有效数据，请检查 ROOT_DIR 路径是否正确。")
        
    return df

# 执行提取
df_results = process_experiment_data(ROOT_DIR)

# 预览前 5 行数据
if not df_results.empty:
    display(df_results.head())

🔍 寻找到 409 个 pickle 文件，开始解析...


Parsing Files: 100%|██████████| 409/409 [00:00<00:00, 2824.47it/s]


✅ 数据处理完毕！共提取 309 条有效记录。
💾 已保存至: ./result_98_2/experiment_results_PowerBI.csv


,Task,Ground_Size,Seed,Algorithm,Strategy,Heuristic,Sorting,UB,D,Budget,Alpha,Model,Objective_f(S),Cost_c(S),Time_s,Node_Count,Open_List_Count,TLE,Solution_Set_Size
21,adult,111,0,EfficientBFS,density_gap,ub2,d,None,None,6.0,0.95,AdultIncomeFeatureSelection,7.206555,5.836918,29.113479,5,1.0,False,5
46,adult,111,0,EfficientBFS,fullbab,ub2,d,None,None,6.0,0.95,AdultIncomeFeatureSelection,7.206555,5.836918,37.456656,12,1.0,False,5
71,adult,111,0,EfficientBFS,look_ahead,ub2,d,None,None,6.0,0.95,AdultIncomeFeatureSelection,7.206555,5.836918,30.124027,4,1.0,False,5
96,adult,111,0,EfficientBFS,traditional,ub2,d,None,None,6.0,0.95,AdultIncomeFeatureSelection,7.206555,5.836918,32.139664,5,1.0,False,5
22,adult,111,0,EfficientBFS,density_gap,ub2,d,None,None,7.0,0.95,AdultIncomeFeatureSelection,7.206555,6.999571,59.563825,22,1.0,False,6
